In [0]:
from pyspark.sql.functions import col

from pyspark.sql.functions import coalesce
from pyspark.sql.functions import try_to_date

df = spark.table("medical_data.silver.patients_s")

df = df.withColumn("birthdate", coalesce(try_to_date("birthdate_", "yyyy-MM-dd"),try_to_date("birthdate_", "dd-MM-yyyy"))) \
    .withColumn("death_date", coalesce(try_to_date("death_date", "yyyy-MM-dd"),  try_to_date("death_date", "dd-MM-yyyy"))) \
    .withColumn("zip_code", col("zip_code").cast("string")) \
    .withColumn("lat", col("lat").cast("double")) \
    .withColumn("lon", col("lon").cast("double"))


df = df.fillna({
    "maiden": "NA",
    "suffix": "NA",
    "prefix": "NA"
})

df = df.dropDuplicates(["id"])

from pyspark.sql.functions import lower, trim

df = df.withColumn("gender", lower(trim(col("gender")))) \
    .withColumn("race", lower(trim(col("race")))) \
    .withColumn("ethnicity", lower(trim(col("ethnicity")))) \
    .withColumn("marital_status", lower(trim(col("marital_status"))))

df = df.filter(col("id").isNotNull())


from pyspark.sql.functions import regexp_replace

df = df.withColumn("first", regexp_replace("first", r"\d+", "")) \
    .withColumn("last", regexp_replace("last", r"\d+", "")) \
    .withColumn("maiden", regexp_replace("maiden", r"\d+", ""))

df = df.withColumn("county", regexp_replace("county", r"(?i)\s*county\s*$", ""))

from pyspark.sql.functions import regexp_extract, when, length

df = df.withColumn("birth_place", regexp_replace("birth_place", r"^(\w+)\s+\1\s+", "$1 "))

df.write.mode("overwrite").saveAsTable("medical_data.silver.patients_s")


# display(df.summary())